### Before start

```Bash
conda activate genome_env

# conda install -n genome_env -c conda-forge -c bioconda plink2 multiqc mosdepth
# conda install -n genome_env -c bioconda bwa
```

### Installed libs

```Bash
bcftools                     1.24             h118bc1c_2            bioconda
bwa                          0.7.19           h577a1d6_1            bioconda
bwa-mem2                     2.3              he70b90d_0            bioconda
bfastp                        1.3.7            h43da1c4_0            bioconda
fastqc                       0.12.1           hdfd78af_0            bioconda
gatk4                        4.6.2.0          py310hdfd78af_1       bioconda
htslib                       1.24             ha79157c_0            bioconda
mosdepth                     0.3.14           h87be163_2            bioconda
multiqc                      1.35             pyhdfd78af_2          bioconda
openjdk                      25.0.2           h1602c4f_26           conda-forge
plink                        1.90b7.7         h18e278d_1            bioconda
plink2                       2.0.0a.7.8       hea6131b_0            bioconda
samtools                     1.24             h9dcdb79_1            bioconda
``

In [2]:
%%bash

conda list -n genome_env | grep plink

E="/home/flavio/miniforge3/envs/genome_env"

echo "== binaries present =="
for t in plink plink2 fastqc fastp samtools bcftools bwa-mem2 gatk multiqc mosdepth tabix bgzip java python; do
  p="$E/bin/$t"
  printf "%-10s %s\n" "$t" "$([ -x "$p" ] && echo present || echo MISSING)"
done

echo
echo "== versions (run unactivated, via absolute path) =="
if [ -x "$E/bin/plink" ]; then "$E/bin/plink" --version 2>&1 | head -1; else echo "plink: cannot execute"; fi
if [ -x "$E/bin/samtools" ]; then "$E/bin/samtools" --version 2>&1 | head -1; else echo "samtools: cannot execute"; fi

plink                        1.90b7.7         h18e278d_1            bioconda
plink2                       2.0.0a.7.8       hea6131b_0            bioconda
== binaries present ==
plink      present
plink2     present
fastqc     present
fastp      present
samtools   present
bcftools   present
bwa-mem2   present
gatk       present
multiqc    present
mosdepth   present
tabix      present
bgzip      present
java       present
python     present

== versions (run unactivated, via absolute path) ==
PLINK v1.9.0-b.8 64-bit (22 Oct 2024)
samtools 1.24


### PLINK & GWAS

- QC

https://plink.readthedocs.io/en/latest/GWAS/

- Regression

https://www.cog-genomics.org/plink/2.0/tutorials/qwas1

- Paper

https://pubmed.ncbi.nlm.nih.gov/23756892/

Rentería ME, Cortes A, Medland SE. Using PLINK for Genome-Wide Association Studies (GWAS) and data analysis. Methods Mol Biol. 2013;1019:193-213. doi: 10.1007/978-1-62703-447-0_8. PMID: 23756892.

### query: Reed 2015 "guide to genome-wide association analysis" PennCATH GWAStutorial.bed clinical.csv

1. A guide to genome‐wide association analysis and post‐analytic interrogation - Reed - 2015 - Statistics in Medicine - Wil
https://onlinelibrary.wiley.com/doi/10.1002/sim.6605

2. A guide to genome-wide association analysis and post-analytic interrogation
https://www.researchgate.net/publication/281588338_A_guide_to_genome-wide_association_analysis_and_post-analytic_interrogation

3. A tutorial on conducting genome‐wide association studies: Quality control and statistical analysis - PMC
https://pmc.ncbi.nlm.nih.gov/articles/PMC6001694/

4. GitHub - monogenea/GWAStutorial: Genome-wide association studies tutorial 🧬
https://github.com/monogenea/GWAStutorial

5. A guide to genome-wide association analysis and post-analytic interrogation - PubMed
https://pubmed.ncbi.nlm.nih.gov/26343929/

6. Multimarker genetic association tests for panel count data
https://arxiv.org/pdf/2609.03113

7. Efficient testing and effect size estimation for set-based genetic association inference via semiparametric multilevel m
https://arxiv.org/pdf/2003.05611

9. Google Scholar
https://scholar.google.com/scholar_lookup?doi=10.1002/sim.6605&amp=&pmid=26343929

10. GWAS tutorial: An Introduction (2015 - out of date)
https://pbreheny.github.io/adv-gwas-tutorial/

In [3]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path

ROOT0 = Path("/home/flavio/uv/gwas/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)
sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from scripts.qc_penncath_lib import *
from scripts.gwaslib import *


In [4]:
root_data = Path('/home/flavio/uv/gwas/data')
root_cath = root_data / 'PennCATH'
root_raw = root_cath / 'raw'

os.listdir(str(root_cath)), os.listdir(str(root_raw))

(['figures', 'raw', 'results', 'interim'],
 ['penncath.bed',
  'penncath.fam',
  'penncath.csv',
  'penncath.tar.gz',
  'penncath.bim'])

### Project dirs

```Bash
PROJECT_ROOT  /home/flavio/uv/gwas
NOTEBOOKS     /home/flavio/uv/gwas/notebooks
DATASET       PennCATH
RAW           /home/flavio/uv/gwas/data/PennCATH/raw
INTERIM       /home/flavio/uv/gwas/data/PennCATH/interim
RESULTS       /home/flavio/uv/gwas/data/PennCATH/results
FIGURES       /home/flavio/uv/gwas/data/PennCATH/figures
LOGS          /home/flavio/uv/gwas/logs

== src tree ==
src
src/assets
src/libs
src/scripts
src/smart
```

In [5]:
gwaslib.ALL_DIRS

[PosixPath('/home/flavio/uv/gwas/data/PennCATH/raw'),
 PosixPath('/home/flavio/uv/gwas/data/reference'),
 PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim'),
 PosixPath('/home/flavio/uv/gwas/data/PennCATH/results'),
 PosixPath('/home/flavio/uv/gwas/data/PennCATH/figures'),
 PosixPath('/home/flavio/uv/gwas/logs')]

In [6]:
# gwaslib.__dict__

In [7]:
QC_GENO = gwaslib.QC["geno"]
QC_MIND = gwaslib.QC["mind"]
QC_MAF = gwaslib.QC["maf"]
QC_HWE = gwaslib.QC["hwe"]

QC_GENO

0.1

In [8]:
gwaslib.DATASET

'PennCATH'

In [9]:
os.environ["GWAS_CONDA_PREFIX"] = "/home/flavio/miniforge3/envs/genome_env"
os.environ["PLINK"] = "/home/flavio/miniforge3/envs/genome_env/bin/plink"
os.environ["PLINK2"] = "/home/flavio/miniforge3/envs/genome_env/bin/plink2"

In [10]:
CONDA_PREFIX_ENV = os.environ.get("GWAS_CONDA_PREFIX")
CONDA_PREFIX_ENV

'/home/flavio/miniforge3/envs/genome_env'

In [11]:
gwaslib.PLINK = os.environ["PLINK"]
gwaslib.PLINK2 = os.environ["PLINK2"]

In [12]:
gwaslib.setup_logging("qc_penncath")

for directory in gwaslib.ALL_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

gwaslib.require(gwaslib.PLINK)
gwaslib.require(gwaslib.PLINK2)

13:32:42 INFO    logging to /home/flavio/uv/gwas/logs/qc_penncath.log


'/home/flavio/miniforge3/envs/genome_env/bin/plink2'

### Overview of PLINK Binary Files 

see: [GWAStic](https://academic.oup.com/bioinformaticsadvances/article/4/1/vbae177/7896157)

When performing a GWAS, genotype data is frequently stored in a binary PLINK format consisting of three core files that must be kept together: [1, 2, 4] 

* .bed file: The binary core genotype file containing the actual genetic variants (SNPs) for all samples. It is the largest file in the set.
* .bim file: A text file with variant information, including the chromosome, SNP ID, genetic distance, base-pair position, and alleles.
* .fam file: A text file containing sample information, including family ID, individual ID, paternal ID, maternal ID, sex, and phenotype. [1, 2, 3] 


![gwas_model](../pictures/gwas_model.jpeg)

Ref: Introducing GWAStic: a user-friendly, cross-platform solution for genome-wide association studies and genomic prediction.  
Authors: Stefanie Lück , Uwe Scholz , Dimitar Douchkov. Bioinformatics Advances, Volume 4, Issue 1, 2024, vbae177, https://doi.org/10.1093/bioadv/vbae177

### BED binary file

A .bed file in a GWAS (Genome-Wide Association Study) analysis is a binary genotype data file used by PLINK. [1, 2, 3]

Do not confuse the binary PLINK .bed genotype file with a standard UCSC BED format text file. [1, 5] 

- PLINK .bed: Binary file encoding individual genotype calls (0, 1, 2, or missing) for genetic association testing.
- UCSC .BED: Tab-separated text file specifying genomic regions (chromosomes, start, and end coordinates) for functional annotation or visualization. [1, 3, 5] 

[1] [https://cloufield.github.io](https://cloufield.github.io/GWASTutorial/03_Data_formats/)  
[2] [https://www.youtube.com](https://www.youtube.com/watch?v=lFlWyltF5-s&t=411)  
[3] [https://pmc.ncbi.nlm.nih.gov](https://pmc.ncbi.nlm.nih.gov/articles/PMC5019244/)  
[4] [https://academic.oup.com](https://academic.oup.com/bioinformaticsadvances/article/4/1/vbae177/7896157)  
[5] [https://en.wikipedia.org](https://en.wikipedia.org/wiki/BED_%28file_format%29)  


In [13]:
filename_bed = gwaslib.ROOT_RAW / "penncath.bed"
filename_bed.exists()

True

### FAM file

A FAM file in a Genome-Wide Association Study (GWAS) is a standard tab- or space-delimited text file containing sample information and pedigree data, used alongside .bed and .bim files as part of the binary [PLINK](https://www.cog-genomics.org/plink/) format. [1, 2] 
## Structure of a FAM File
A .fam file has no header line and contains one row per sample with 6 fixed columns: [2, 3] 

   1. Family ID (FID): Identifies the family or group.
   2. Within-family Individual ID (IID): Unique sample or individual identifier.
   3. Paternal ID (PAT): ID of the father (0 if missing/unspecified).
   4. Maternal ID (MAT): ID of the mother (0 if missing/unspecified).
   5. Sex (SEX): 1 = male, 2 = female, other/0 = unknown.
   6. Phenotype (PHENOTYPE): The trait value. Typically coded as -9 or 0 for missing, 1 for control/unaffected, and 2 for case/affected (for binary traits), or a continuous numeric value for quantitative traits. [2, 3] 


[1] [https://pmc.ncbi.nlm.nih.gov](https://pmc.ncbi.nlm.nih.gov/articles/PMC6001694/)  
[2] [https://www.youtube.com](https://www.youtube.com/watch?v=lFlWyltF5-s&t=411)  
[3] [https://www.cog-genomics.org](https://www.cog-genomics.org/plink/1.9/formats)  


In [14]:
print(RAW_STEM)
os.listdir(str(gwaslib.ROOT_RAW))

/home/flavio/uv/gwas/data/PennCATH/raw/penncath


['penncath.bed',
 'penncath.fam',
 'penncath.csv',
 'penncath.tar.gz',
 'penncath.bim']

In [15]:
record("00_raw", RAW_STEM)
pheno, covar, idmap, clinical = build_phenotype()

13:32:43 INFO    [00_raw] 861473 variants x 1401 samples
13:32:44 INFO    1401 of 1401 sample IDs carry literal quote characters
13:32:45 INFO    phenotype: 933 cases, 468 controls
13:32:45 INFO    sex .fam vs clinical: 0 disagreements (no X chromosome, so --check-sex is not available)
13:32:45 INFO    covariate missingness: {'age': 0, 'tg': 92, 'hdl': 92, 'ldl': 119}


In [16]:
pheno

PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/penncath.pheno')

In [17]:
covar

PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/penncath.covar')

In [18]:
idmap

PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/penncath.update_ids')

In [19]:
n_case = int((clinical["CAD"] == 1).sum())
n_ctrl = int((clinical["CAD"] == 0).sum())

f"There are {n_case} cases and {n_ctrl} controls."

'There are 933 cases and 468 controls.'

### plink --make-bed

01a - strip the quote characters out of the sample IDs, once, so that every ID file written from here on matches what PLINK sees.  
This rewrite also resolves the 26,154 '-9' allele codes that otherwise block --maf.

#### Log

```Bash
PLINK v1.9.0-b.8 64-bit (22 Oct 2024)  
Options in effect:  
  --bfile /home/flavio/uv/gwas/data/PennCATH/raw/penncath  
  --make-bed  
  --out /home/flavio/uv/gwas/data/PennCATH/interim/01a_ids  
  --update-ids /home/flavio/uv/gwas/data/PennCATH/interim/penncath.update_ids  

861473 variants loaded from .bim file.  
1401 people (937 males, 464 females) loaded from .fam.  
933 phenotype values loaded from .fam.  
--update-ids: 1401 people updated.  
```

In [20]:
idmap

PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/penncath.update_ids')

In [21]:
s01a = gwaslib.plink(["--bfile", RAW_STEM, "--update-ids", idmap, "--make-bed"], out=gwaslib.ROOT_INTERIM / "01a_ids")
record("01a_ids_unquoted", s01a)

13:32:45 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/raw/penncath --update-ids /home/flavio/uv/gwas/data/PennCATH/interim/penncath.update_ids --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/01a_ids
13:32:46 INFO    ok (0.7s)
13:32:47 INFO    [01a_ids_unquoted] 861473 variants x 1401 samples


### 01b - attach the phenotype, then verify it actually landed.

In [22]:
prefix_pheno = gwaslib.ROOT_INTERIM / "01b_pheno"

s01 = gwaslib.plink(["--bfile", s01a, "--pheno", pheno, "--pheno-name", "CAD", "--make-bed"], out=prefix_pheno)
record("01b_phenotype", s01)
assert_phenotype(s01, n_case, n_ctrl)
print(f"Result in {gwaslib.ROOT_INTERIM}, files {s01} - {n_case} cases, {n_ctrl} controls")

13:32:47 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/01a_ids --pheno /home/flavio/uv/gwas/data/PennCATH/interim/penncath.pheno --pheno-name CAD --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/01b_pheno
13:32:48 INFO    ok (0.6s)
13:32:48 INFO    [01b_phenotype] 861473 variants x 1401 samples
13:32:48 INFO    phenotype in .fam: 933 cases, 468 controls, 0 missing
Result in /home/flavio/uv/gwas/data/PennCATH/interim, files /home/flavio/uv/gwas/data/PennCATH/interim/01b_pheno - 933 cases, 468 controls


### Reporting runs on the un-filtered data, so the figures show what the thresholds are about to remove.

In [23]:
gwaslib.plink(["--bfile", s01, "--missing"], out=gwaslib.ROOT_INTERIM / "rep_missing")
gwaslib.plink(["--bfile", s01, "--freq"], out=gwaslib.ROOT_INTERIM / "rep_freq")
gwaslib.plink(["--bfile", s01, "--hardy"], out=gwaslib.ROOT_INTERIM / "rep_hardy")

13:32:48 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/01b_pheno --missing --out /home/flavio/uv/gwas/data/PennCATH/interim/rep_missing
13:32:49 INFO    ok (0.4s)
13:32:49 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/01b_pheno --freq --out /home/flavio/uv/gwas/data/PennCATH/interim/rep_freq
13:32:49 INFO    ok (0.3s)
13:32:49 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/01b_pheno --hardy --out /home/flavio/uv/gwas/data/PennCATH/interim/rep_hardy
13:32:50 INFO    ok (0.7s)


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/rep_hardy')

### 02 - call rate.  Variants first, then samples

- dropping bad variants improves the per-sample rates, so fewer samples are lost unnecessarily.

In [24]:
s02a = gwaslib.plink(["--bfile", s01, "--geno", QC_GENO, "--make-bed"], out=gwaslib.ROOT_INTERIM / "02a_geno")
record(f"02a_geno<{QC_GENO}", s02a)
s02a

13:32:50 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/01b_pheno --geno 0.1 --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/02a_geno
13:32:50 INFO    ok (0.6s)
13:32:51 INFO    [02a_geno<0.1] 814081 variants x 1401 samples


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/02a_geno')

In [25]:
s02 = gwaslib.plink(["--bfile", s02a, "--mind", QC_MIND, "--make-bed"], out=gwaslib.ROOT_INTERIM / "02b_mind")
record(f"02b_mind<{QC_MIND}", s02)

13:32:51 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/02a_geno --mind 0.1 --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/02b_mind
13:32:52 INFO    ok (0.7s)
13:32:53 INFO    [02b_mind<0.1] 814081 variants x 1401 samples


### 03 - MAF and HWE.  

For case/control data PLINK 1.9 applies --hwe to controls only, which is what we want: a true association can push cases out of equilibrium at the causal locus.

In [26]:
s03a = gwaslib.plink(["--bfile", s02, "--maf", QC_MAF, "--make-bed"], out=gwaslib.ROOT_INTERIM / "03a_maf")
record(f"03a_maf>{QC_MAF}", s03a)
s03a

13:32:53 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/02b_mind --maf 0.01 --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/03a_maf
13:32:53 INFO    ok (0.6s)
13:32:54 INFO    [03a_maf>0.01] 697479 variants x 1401 samples


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/03a_maf')

In [27]:
s03 = gwaslib.plink(["--bfile", s03a, "--hwe", QC_HWE, "midp", "--make-bed"], out=gwaslib.ROOT_INTERIM / "03b_hwe")
record(f"03b_hwe>{QC_HWE:g}", s03)
s03

13:32:54 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/03a_maf --hwe 1e-10 midp --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/03b_hwe
13:32:55 INFO    ok (0.7s)
13:32:56 INFO    [03b_hwe>1e-10] 696660 variants x 1401 samples


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/03b_hwe')

### linkage disequilibrium (LD) pruning

Heterozygosity, relatedness and PCA are all distorted by correlated markers, so they use this pruned subset.

In [28]:
prune = gwaslib.plink(["--bfile", s03, "--indep-pairwise",
                      gwaslib.QC["ld_window"], gwaslib.QC["ld_step"], gwaslib.QC["ld_r2"]],
                      out=gwaslib.ROOT_INTERIM / "pruned")
n_pruned = sum(1 for _ in open(f"{prune}.prune.in"))
gwaslib.LOG.info("LD pruning kept %d independent variants", n_pruned)
prune

13:32:56 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/03b_hwe --indep-pairwise 50 5 0.2 --out /home/flavio/uv/gwas/data/PennCATH/interim/pruned
13:32:56 INFO    ok (0.9s)
13:32:57 INFO    LD pruning kept 116591 independent variants


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/pruned')

### 04 - heterozygosity outliers.  

Excess heterozygosity suggests sample contamination, a deficit suggests inbreeding or poor DNA quality.

In [29]:
het_stem = gwaslib.plink(["--bfile", s03, "--extract", f"{prune}.prune.in", "--het"], out=gwaslib.ROOT_INTERIM / "rep_het")

df_het = pd.read_csv(f"{het_stem}.het", sep=r"\s+", quoting=csv.QUOTE_NONE)
mean_f, sd_f = df_het["F"].mean(), df_het["F"].std()
limit = gwaslib.QC["het_sd"] * sd_f
df_het["het_outlier"] = (df_het["F"] - mean_f).abs() > limit
n_outliers = int(df_het["het_outlier"].sum())
remove = gwaslib.ROOT_INTERIM / "het_outliers.remove"

df_het.loc[df_het["het_outlier"], ["FID", "IID"]].to_csv(remove, sep="\t", index=False, header=False, quoting=csv.QUOTE_NONE)

gwaslib.LOG.info("heterozygosity F mean %.4f sd %.4f -> %d outliers beyond %.1f SD", mean_f, sd_f, n_outliers, gwaslib.QC["het_sd"])

print(het_stem)
print(df_het.shape)
df_het.head(3)

13:32:57 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/03b_hwe --extract /home/flavio/uv/gwas/data/PennCATH/interim/pruned.prune.in --het --out /home/flavio/uv/gwas/data/PennCATH/interim/rep_het
13:32:57 INFO    ok (0.3s)
13:32:57 INFO    heterozygosity F mean -0.0065 sd 0.0147 -> 19 outliers beyond 3.0 SD
/home/flavio/uv/gwas/data/PennCATH/interim/rep_het
(1401, 7)


,FID,IID,O(HOM),E(HOM),N(NM),F,het_outlier
0,10002,1,79742,80950.0,113297,-0.03724,False
1,10004,1,81167,81820.0,114466,-0.02002,False
2,10005,1,81554,82150.0,115042,-0.01818,False


In [30]:
s04 = gwaslib.plink(["--bfile", s03, "--remove", remove, "--make-bed"], out=gwaslib.ROOT_INTERIM / "04_het")
record("04_het_outliers_removed", s04)
dropped = STAGES[-2][2] - STAGES[-1][2]
if dropped != n_outliers:
    raise ValueError(f"--remove dropped {dropped} samples, expected {n_outliers}")


print(f"Dropped {dropped} samples")
s04

13:32:57 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/03b_hwe --remove /home/flavio/uv/gwas/data/PennCATH/interim/het_outliers.remove --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/04_het
13:32:58 INFO    ok (0.6s)
13:32:59 INFO    [04_het_outliers_removed] 696660 variants x 1382 samples
Dropped 19 samples


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/04_het')

### 05 - relatedness on the pruned SNP set.  

### KING kinship is robust to population structure, which IBD-based --genome is not.

--king-cutoff writes the sample lists only; the fileset is then subset with --keep so the full variant set is preserved.

In [31]:
kin = gwaslib.plink(["--bfile", s04, "--extract", f"{prune}.prune.in", "--make-king-table", "--king-table-filter", gwaslib.QC["king_cutoff"]],
                    out=gwaslib.ROOT_INTERIM / "rep_king", v2=True)

df_kin = pd.read_csv(f"{kin}.kin0", sep="\t")

gwaslib.LOG.info("%d sample pairs with kinship > %.4f", len(df_kin), gwaslib.QC["king_cutoff"])

if len(df_kin):
    df_kin.to_csv(gwaslib.ROOT_RESULTS / "related_pairs.csv", index=False)

king = gwaslib.plink(["--bfile", s04, "--extract", f"{prune}.prune.in",
                "--king-cutoff", gwaslib.QC["king_cutoff"]],
                out=gwaslib.ROOT_INTERIM / "05_king", v2=True)

s05 = gwaslib.plink(["--bfile", s04, "--keep", f"{king}.king.cutoff.in.id", "--make-bed"], out=gwaslib.ROOT_INTERIM / "05_unrelated", v2=True)
record("05_unrelated", s05)

13:40:27 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink2 --bfile /home/flavio/uv/gwas/data/PennCATH/interim/04_het --extract /home/flavio/uv/gwas/data/PennCATH/interim/pruned.prune.in --make-king-table --king-table-filter 0.0884 --out /home/flavio/uv/gwas/data/PennCATH/interim/rep_king
13:40:27 INFO    ok (0.6s)
13:40:27 INFO    0 sample pairs with kinship > 0.0884
13:40:27 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink2 --bfile /home/flavio/uv/gwas/data/PennCATH/interim/04_het --extract /home/flavio/uv/gwas/data/PennCATH/interim/pruned.prune.in --king-cutoff 0.0884 --out /home/flavio/uv/gwas/data/PennCATH/interim/05_king
13:40:28 INFO    ok (0.5s)
13:40:28 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink2 --bfile /home/flavio/uv/gwas/data/PennCATH/interim/04_het --keep /home/flavio/uv/gwas/data/PennCATH/interim/05_king.king.cutoff.in.id --make-bed --out /home/flavio/uv/gwas/data/PennCATH/interim/05_unrelated
13:40:28 INFO    ok (0.3s)
13:40:29 INFO

### PCA on the unrelated, pruned set. 

With a single-ancestry cohort this is an outlier screen and a source of covariates, not an ancestry assignment.

In [34]:
pca = gwaslib.plink(["--bfile", s05, "--pca", gwaslib.QC["n_pcs"]], out=gwaslib.ROOT_RESULTS / "penncath_pca", v2=True)

# Final analysis-ready fileset: all QC-passing samples, all QC-passing variants
#  (not just the pruned subset, which exists only for the steps
# above).
final = gwaslib.plink(["--bfile", s03, "--keep", f"{s05}.fam", "--make-bed"], out=gwaslib.ROOT_RESULTS / "penncath_qc")
record("06_final", final)

13:42:46 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink2 --bfile /home/flavio/uv/gwas/data/PennCATH/interim/05_unrelated --pca 10 --out /home/flavio/uv/gwas/data/PennCATH/results/penncath_pca
13:43:29 INFO    ok (43.1s)
13:43:29 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/interim/03b_hwe --keep /home/flavio/uv/gwas/data/PennCATH/interim/05_unrelated.fam --make-bed --out /home/flavio/uv/gwas/data/PennCATH/results/penncath_qc
13:43:30 INFO    ok (0.6s)
13:43:30 INFO    [06_final] 696660 variants x 1382 samples


In [37]:
write_summary()

13:43:38 INFO    wrote /home/flavio/uv/gwas/data/PennCATH/results/qc_summary.csv
                  stage  variants  samples  variants_lost  samples_lost
                 00_raw    861473     1401              0             0
       01a_ids_unquoted    861473     1401              0             0
          01b_phenotype    861473     1401              0             0
           02a_geno<0.1    814081     1401          47392             0
           02b_mind<0.1    814081     1401              0             0
           03a_maf>0.01    697479     1401         116602             0
          03b_hwe>1e-10    696660     1401            819             0
04_het_outliers_removed    696660     1382              0            19
           05_unrelated    696660     1382              0             0
               06_final    696660     1382              0             0


In [36]:
make_figures(clinical, pca)

13:43:38 INFO    wrote /home/flavio/uv/gwas/data/PennCATH/figures/penncath_qc.png


In [38]:
gwaslib.ROOT_FIGURES / "penncath_qc.png"

PosixPath('/home/flavio/uv/gwas/data/PennCATH/figures/penncath_qc.png')

![penncath_qc](/home/flavio/uv/gwas/data/PennCATH/figures/penncath_qc.png)